# 03 — CatBoost categorical-aware model

**Owners:** Midhun / Ajmeer
**Role:** primary mixed-type deployment candidate

Interview explanation: *“CatBoost is designed for categorical tabular data. Its
ordered categorical statistics reduce target leakage and let us retain card, email,
device, and address identities without producing a huge one-hot matrix.”*


## Feature engineering for this approach

- Numeric quantities remain numeric with `NaN`; no scaling is needed.
- Text categories and numeric identifier codes become categorical strings.
- Missing categorical values become the explicit label `MISSING`.
- High cardinality is handled natively by CatBoost's ordered categorical statistics.
- We do not add manual target encoding, avoiding duplicate complexity and leakage risk.
- Class weights are calculated using the training period only.


In [ ]:
from pathlib import Path
_install_root = Path.cwd().resolve()
for _candidate in [_install_root, *_install_root.parents]:
    if (_candidate / "requirements-training.txt").exists():
        _requirements = _candidate / "requirements-training.txt"
        break
else:
    raise FileNotFoundError("Open this notebook from inside the cloned repository")
%pip install -q -r {_requirements}


In [ ]:
from pathlib import Path
import gc, json, os, sys, time
import numpy as np
import pandas as pd

def locate_project_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / ".git").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Run this notebook from inside the cloned repository")

PROJECT_ROOT = locate_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)


In [ ]:
required = [
    PROCESSED_DIR / "train.parquet",
    PROCESSED_DIR / "validation.parquet",
    PROCESSED_DIR / "test.parquet",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Run 00_shared_data_preparation.ipynb first. Missing: " + ", ".join(missing)
    )

train = pd.read_parquet(required[0])
validation = pd.read_parquet(required[1])
test = pd.read_parquet(required[2])

def stratified_debug_sample(frame, rows):
    if rows is None or rows >= len(frame):
        return frame
    return (
        frame.groupby("isFraud", group_keys=False)
        .apply(lambda group: group.sample(
            n=max(1, round(rows * len(group) / len(frame))),
            random_state=RANDOM_SEED,
        ), include_groups=True)
        .sort_values(["TransactionDT", "TransactionID"])
        .reset_index(drop=True)
    )

FAST_RUN = False  # Set True only to verify the notebook; never report these metrics.
if FAST_RUN:
    train = stratified_debug_sample(train, 60_000)
    validation = stratified_debug_sample(validation, 20_000)
    test = stratified_debug_sample(test, 20_000)

TARGET = "isFraud"
DROP_FROM_MODEL = ["isFraud", "TransactionID"]
X_train, y_train = train.drop(columns=DROP_FROM_MODEL), train[TARGET].astype("int8")
X_validation, y_validation = validation.drop(columns=DROP_FROM_MODEL), validation[TARGET].astype("int8")
X_test, y_test = test.drop(columns=DROP_FROM_MODEL), test[TARGET].astype("int8")

print("Train:", X_train.shape, "fraud rate:", f"{y_train.mean():.4%}")
print("Validation:", X_validation.shape, "fraud rate:", f"{y_validation.mean():.4%}")
print("Test:", X_test.shape, "fraud rate:", f"{y_test.mean():.4%}")


In [ ]:
MODEL_KEY = "catboost"


In [ ]:
from datetime import datetime, timezone
from src.fraud_pipeline.artifacts import build_manifest, package_versions, write_json
from src.fraud_pipeline.evaluation import evaluate_binary_classifier, select_operating_threshold

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = ARTIFACT_ROOT / MODEL_KEY / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("This run will be saved to:", RUN_DIR)


## Fit the stable CatBoost input contract

The transformation records exact feature order and categorical names. This object
will later transform the same common API input.


In [ ]:
import joblib, torch
from catboost import CatBoostClassifier, Pool
from src.fraud_pipeline.preprocessing import CatBoostPreprocessor

preprocessor = CatBoostPreprocessor().fit(X_train)
X_train_model = preprocessor.transform(X_train)
X_validation_model = preprocessor.transform(X_validation)
print("Features:", X_train_model.shape[1])
print("Categorical features:", len(preprocessor.categorical_features))


## Train with early stopping

Set `USE_GPU=False` if the Lightning Studio is on CPU. GPU CatBoost is faster but
can produce very small numerical differences from CPU runs.


In [ ]:
USE_GPU = torch.cuda.is_available()
negative, positive = np.bincount(y_train)
class_weight = float(negative / positive)

train_pool = Pool(X_train_model, y_train, cat_features=preprocessor.categorical_features)
validation_pool = Pool(X_validation_model, y_validation, cat_features=preprocessor.categorical_features)
catboost_parameters = dict(
    iterations=4000,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    class_weights=[1.0, class_weight],
    l2_leaf_reg=5.0,
    random_seed=RANDOM_SEED,
    task_type="GPU" if USE_GPU else "CPU",
    allow_writing_files=False,
    verbose=100,
)
if USE_GPU:
    catboost_parameters["devices"] = "0"
model = CatBoostClassifier(**catboost_parameters)
started = time.perf_counter()
model.fit(train_pool, eval_set=validation_pool, early_stopping_rounds=200, use_best_model=True)
training_seconds = time.perf_counter() - started
print("Best iteration:", model.get_best_iteration())


## Validation threshold and final holdout evaluation


In [ ]:
validation_probability = model.predict_proba(validation_pool)[:, 1]
threshold_record = select_operating_threshold(y_validation, validation_probability, minimum_precision=0.10)
threshold = float(threshold_record["threshold"])
validation_metrics = evaluate_binary_classifier(y_validation, validation_probability, threshold)

del train_pool, X_train_model
gc.collect()
X_test_model = preprocessor.transform(X_test)
test_pool = Pool(X_test_model, y_test, cat_features=preprocessor.categorical_features)
started = time.perf_counter()
test_probability = model.predict_proba(test_pool)[:, 1]
prediction_seconds = time.perf_counter() - started
test_metrics = evaluate_binary_classifier(y_test, test_probability, threshold)
display(pd.DataFrame([validation_metrics, test_metrics], index=["validation", "test"])[["pr_auc", "roc_auc", "precision", "recall", "f1", "brier_score"]])


## Importance, native `.cbm` model, and supporting contract


In [ ]:
importance = pd.DataFrame({
    "feature": preprocessor.feature_columns,
    "importance": model.get_feature_importance(validation_pool),
}).sort_values("importance", ascending=False)
importance.head(100).to_csv(RUN_DIR / "feature_importance.csv", index=False)

model_path = RUN_DIR / "model.cbm"
model.save_model(str(model_path), format="cbm")
preprocessor_path = RUN_DIR / "preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path, compress=3)
pd.DataFrame({"TransactionID": validation["TransactionID"], "isFraud": y_validation, "probability": validation_probability}).to_parquet(RUN_DIR / "validation_predictions.parquet", index=False)
pd.DataFrame({"TransactionID": test["TransactionID"], "isFraud": y_test, "probability": test_probability}).to_parquet(RUN_DIR / "test_predictions.parquet", index=False)
write_json(RUN_DIR / "threshold.json", threshold_record)
write_json(RUN_DIR / "metrics.json", {"validation": validation_metrics, "test": test_metrics})
write_json(RUN_DIR / "feature_schema.json", {"model": MODEL_KEY, "feature_columns": preprocessor.feature_columns, "categorical_features": preprocessor.categorical_features})
write_json(RUN_DIR / "training_config.json", {
    "model": MODEL_KEY, "run_id": RUN_ID, "random_seed": RANDOM_SEED,
    "fast_run": FAST_RUN, "training_seconds": training_seconds,
    "test_prediction_seconds": prediction_seconds, "use_gpu": USE_GPU,
    "best_iteration": model.get_best_iteration(), "class_weight": class_weight,
    "parameters": model.get_params(),
    "versions": package_versions(["numpy", "pandas", "catboost", "joblib"]),
})


## Mandatory reload test


In [ ]:
loaded_preprocessor = joblib.load(preprocessor_path)
loaded_model = CatBoostClassifier()
loaded_model.load_model(str(model_path))
sample = loaded_preprocessor.transform(X_validation.iloc[:5])
sample_pool = Pool(sample, cat_features=loaded_preprocessor.categorical_features)
before = model.predict_proba(validation_pool.slice(list(range(5))))[:, 1]
after = loaded_model.predict_proba(sample_pool)[:, 1]
np.testing.assert_allclose(before, after, rtol=1e-6, atol=1e-8)
write_json(RUN_DIR / "manifest.json", build_manifest(RUN_DIR))
print("Reload test passed:", after)
print("Artifact directory:", RUN_DIR)


In [ ]:
# Optional promotion step: upload this versioned run to a private Cloudflare R2 bucket.
# Create these as Lightning secrets/environment variables; never paste keys into a cell.
UPLOAD_TO_R2 = False

if UPLOAD_TO_R2:
    import boto3
    required_names = [
        "R2_ENDPOINT_URL", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_BUCKET_NAME"
    ]
    absent = [name for name in required_names if not os.getenv(name)]
    if absent:
        raise RuntimeError("Missing Lightning secrets: " + ", ".join(absent))
    client = boto3.client(
        "s3",
        endpoint_url=os.environ["R2_ENDPOINT_URL"],
        aws_access_key_id=os.environ["R2_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["R2_SECRET_ACCESS_KEY"],
        region_name="auto",
    )
    prefix = f"{MODEL_KEY}/{RUN_ID}"
    for local_path in RUN_DIR.rglob("*"):
        if local_path.is_file():
            key = f"{prefix}/{local_path.relative_to(RUN_DIR).as_posix()}"
            client.upload_file(str(local_path), os.environ["R2_BUCKET_NAME"], key)
    print(f"Uploaded to r2://{os.environ['R2_BUCKET_NAME']}/{prefix}/")
else:
    print("R2 upload skipped. Set UPLOAD_TO_R2=True after configuring Lightning secrets.")


## Interview checklist

Be ready to explain ordered categorical statistics, why code-like numeric fields are
strings here, why missing categories are explicit, why CatBoost does not require
scaling, and how the native model plus saved preprocessor reaches the API.
